In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q transformers numpy torch

In [3]:
import torch
import joblib
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = "./drive/MyDrive/l2_models/Лояльность_и_продажи"
MAX_LEN = 256
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(DEVICE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
mlb = joblib.load(f"{MODEL_PATH}/level2_mlb.joblib")
model.eval()

def predict_level2_above_thresh(text, threshold=0.3):
    """
    Возвращает все Level 2 категории с вероятностью выше threshold.
    """
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding='max_length',
        truncation=True,
        max_length=MAX_LEN
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits.squeeze(0)
        probs = torch.sigmoid(logits).cpu().numpy()

    idx_above = np.where(probs > threshold)[0]
    if len(idx_above) == 0:
        idx_above = [np.argmax(probs)]
    labels = np.array(mlb.classes_)[idx_above]
    scores = probs[idx_above]

    result = [(label, round(float(score), 4)) for label, score in zip(labels, scores)]
    result = sorted(result, key=lambda x: x[1], reverse=True)
    return result

if __name__ == "__main__":
    test_texts = [
        "Привет! Подскажите, как узнать сколько миль у меня осталось на карте S7 Priority? Хочу спланировать покупку билета на ближайший вылет.",
        "Здравствуйте! Обращаюсь по поводу акции \"Летим в Сочи\". Не могу понять, какие рейсы участвуют? И как применить промокод при оплате? Спасибо.",
        "Добрый день! У меня сгорело 5000 миль! Почему мне не пришло уведомление? Это просто ужас, я их копил полгода! Требую восстановить.",
        "Скажите, а есть какие-то специальные тарифы для студентов? Летаю часто по учебе, очень нужна скидка. Спасибо за ответ!",
        "Пытаюсь оплатить часть билета милями, но система выдает ошибку. Пишет \"Недостаточно миль\", хотя у меня 15000. Что делать?",
        "Вижу что у вас идет распродажа \"Осенние каникулы\", но цены все равно выше чем у конкурентов. Будет ли еще снижение цен?",
        "Здраствуйте! У меня вопросс по начислению миль за последний перелет Москва-Казань. Должны были начислить 500 миль, но их нету.",
        "Подскажите, в чем разница между тарифами \"Эконом\" и \"Эконом гибкий\"? Хочу понять, стоит ли переплачивать за возможность изменений.",
        "Можно ли как-то ускорить начисление миль? Летел вчера, а мили до сих пор не пришли. Обычно быстрее приходили.",
        "Почему у вас такие дорогие тарифы на бизнес-класс? У Аэрофлота в 1.5 раза дешевле на те же даты. Есть ли какие-то акции?",
        "Хочу передать часть миль сыну для покупки билета. Как это сделать и есть ли комиссия? Спасибо!",
        "Заметила что цены на билеты вечером ниже чем утром. Это какая-то акция или просто совпадение? Когда лучше покупать?",
        "Не приходят уведомления о состоянии счета миль. Проверьте пожалуйста настройки и пришлите историю начислений за последние 3 месяца.",
        "Есть ли у вас программа рассрочки на дорогие билеты? Хочу лететь в отпуск, но сразу всю сумму платить тяжело.",
        "Почему при оплате билета милями списывается больше чем показывалось в калькуляторе? Это обман!",
        "Вижу что у вас есть \"Тариф для своих\", но не понимаю как на него попасть. Нужно быть определенное количество раз полететь?",
        "Как восстановить карту S7 Priority если я ее потерял? И не смогут ли ей воспользоваться другие люди?",
        "Почему на один и тот же рейс такие разные цены в разные дни? Это манипулирование ценами! Хочу понять логику ценообразования.",
        "Можно ли копить мили вместе с семьей? У нас с мужем отдельные карты, но хотим объединить мили для общего путешествия.",
        "Будет ли черная пятница у S7? Жду больших скидок на билеты зарубеж, очень хочется съездить в отпуск.",
        "Почему мне не начисляются мили за перелеты партнерами? Летал Turkish Airlines по коду S7, мили не пришли.",
        "Чем отличается \"бизнес-гибкий\" от обычного бизнеса? Стоит ли переплачивать если даты точно известны?",
        "Мой статус в программе лояльности понизился без предупреждения! Почему? Я регулярно летаю вашими авиалиниями.",
        "Есть ли скрытые платежи в ваших тарифах? При выборе билета одна цена, а при оплате оказывается на 20% дороже.",
        "Как узнать когда истекает срок действия моих миль? Хочу спланировать их использование чтобы ничего не сгорело.",
        "Почему у вас нет семейных тарифов? Летим вчетвером, а покупать отдельные билеты невыгодно. Есть ли групповые скидки?",
        "Не могу войти в аккаунт S7 Priority, пишет неверный пароль. Восстановление не работает, присылает ошибку. Помогите!",
        "Заметила что цены растут если часто ищешь один и тот же рейс. Это специальная система или просто кажется?",
        "Можно ли оплатить милями полностью весь билет или только часть? И есть ли ограничения по направлениям?"
    ]

    for i, text in enumerate(test_texts):
        prediction = predict_level2_above_thresh(text, threshold=0.3)
        print(f"\nТекст {i+1}: {text}")
        print("Категории с вероятностью > 0.3:")
        for label, prob in prediction:
            print(f"  - {label}: {prob}")



Текст 1: Привет! Подскажите, как узнать сколько миль у меня осталось на карте S7 Priority? Хочу спланировать покупку билета на ближайший вылет.
Категории с вероятностью > 0.3:
  - Программа лояльности: 0.9901

Текст 2: Здравствуйте! Обращаюсь по поводу акции "Летим в Сочи". Не могу понять, какие рейсы участвуют? И как применить промокод при оплате? Спасибо.
Категории с вероятностью > 0.3:
  - Акции тарифы: 0.9931

Текст 3: Добрый день! У меня сгорело 5000 миль! Почему мне не пришло уведомление? Это просто ужас, я их копил полгода! Требую восстановить.
Категории с вероятностью > 0.3:
  - Программа лояльности: 0.9903

Текст 4: Скажите, а есть какие-то специальные тарифы для студентов? Летаю часто по учебе, очень нужна скидка. Спасибо за ответ!
Категории с вероятностью > 0.3:
  - Акции тарифы: 0.9941

Текст 5: Пытаюсь оплатить часть билета милями, но система выдает ошибку. Пишет "Недостаточно миль", хотя у меня 15000. Что делать?
Категории с вероятностью > 0.3:
  - Программа лояльности: 